# Bronze to Silver ETL — Health Insurance Marketplace

Este notebook implementa o ETL da **camada Silver** do Data Lake, aplicando limpezas e transformações nos dados brutos da Bronze para prepará-los para análises na Gold.

## Objetivo

- Ler tabelas da camada Bronze (Iceberg via Glue Catalog).
- Aplicar transformações de qualidade de dados baseadas nas recomendações do `bronze_exploration.ipynb`.
- Escrever tabelas limpas na camada Silver em Parquet com compressão Snappy.
- Particionar tabelas grandes (ex: `Rate`) por `StateCode` e `BusinessYear` para otimização de consultas no Athena.

## Transformações Aplicadas

Baseado na seção 12 do `bronze_exploration.ipynb`:

1. **BenefitsCostSharing**: Parsear campos de copay/coinsurance de strings para decimais; mapear "No Charge" → 0.0, "Not Applicable" → null.
2. **BenefitsCostSharing**: Distinguir nulos legítimos em Tier2.
3. **PlanAttributes**: Agrupar variantes CSR pelo `HIOSProductId`.
4. **Rate**: Filtrar valores extremos (IndividualRate > 0 e < 3000).
5. **Crosswalk**: Tratar nulos em PlanId como descontinuações.
6. **Geral**: Deduplicar por `PlanId` + ano mantendo maior `VersionNum`.
7. **Particionamento**: Aplicar particionamento por `StateCode` + `BusinessYear` onde relevante.

## Como executar

1. Abra o projeto no Dev Container.
2. Atualize credenciais AWS se necessário.
3. Selecione kernel Python 3.11.14.
4. Execute sequencialmente.

> **Nota:** Este notebook usa PySpark/Glue para processamento distribuído. Em produção, execute como Glue Job.

In [1]:

import logging
import re
import sys

import awswrangler as wr
import boto3
from awsglue.context import GlueContext
from awsglue.job import Job
from awsglue.utils import getResolvedOptions
from pyspark.conf import SparkConf
from pyspark.context import SparkContext
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType, StringType
from pyspark.sql.window import Window

In [2]:
# ##################################################################################################
# Parâmetros do Job — usados em produção pelo Glue (comentado no ambiente local)
#
# args = getResolvedOptions(
#     sys.argv, ["JOB_NAME", "PIPELINE_NAME", "BRONZE_DATABASE", "SILVER_DATABASE"]
# )
#
# PIPELINE_NAME = args["PIPELINE_NAME"]
# BRONZE_DATABASE = args["BRONZE_DATABASE"]
# SILVER_DATABASE = args["SILVER_DATABASE"]

In [3]:
# ##################################################################################################
# Variáveis locais — substitui os parâmetros do job para execução no notebook

PIPELINE_NAME   = "health_insurance"
BRONZE_DATABASE = "eedb015_bronze"
SILVER_DATABASE = "eedb015_silver"

# Descobre o account ID dinamicamente via STS
_account_id = boto3.client("sts").get_caller_identity()["Account"]

# Limite de tabelas a processar — útil para não sobrecarregar o ambiente local.
# Altere para None para processar todas.
LOCAL_TABLE_LIMIT = None

In [4]:
# ##################################################################################################
# Inicialização Spark / Glue com extensões Iceberg

logging.basicConfig(
    level=logging.INFO,
    format="[%(asctime)s] %(levelname)s — %(message)s",
    datefmt="%Y-%m-%dT%H:%M:%S",
)
logger = logging.getLogger("bronze_to_silver")

scf = SparkConf()
scf.setAll([
    ("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"),
    ("spark.sql.catalog.glue_catalog", "org.apache.iceberg.spark.SparkCatalog"),
    ("spark.sql.catalog.glue_catalog.catalog-impl", "org.apache.iceberg.aws.glue.GlueCatalog"),
    ("spark.sql.catalog.glue_catalog.io-impl", "org.apache.iceberg.aws.s3.S3FileIO"),
    ("spark.sql.defaultCatalog", "glue_catalog"),
    # Parser de datas legado
    ("spark.sql.legacy.timeParserPolicy", "LEGACY"),
    # Otimização de escrita no S3
    ("spark.hadoop.mapreduce.fileoutputcommitter.algorithm.version", "2"),
    ("spark.speculation", "true"),
    # Sobrescrita dinâmica de partições
    ("spark.sql.sources.partitionOverwriteMode", "dynamic"),
])

sc = SparkContext(conf=scf)
glue_ctx = GlueContext(sc)
spark = glue_ctx.spark_session

# Em produção: job = Job(glue_ctx); job.init(args['JOB_NAME'], args)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/usr/local/lib/python3.11/site-packages/pyspark/sql/context.py:113: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


In [5]:
# ##################################################################################################
# Funções auxiliares para transformações

def parse_string_to_double(col_name: str):
    """Converte coluna de string para decimal, tratando percentuais, moeda e "Not Applicable"."""
    column = F.col(col_name)
    cleaned_numeric = F.regexp_replace(column, r"[\$,]", "")
    cleaned_percent = F.regexp_replace(column, r"[%,\s]", "")

    return (
        F.when(column.isin("No Charge", "no charge"), F.lit(0.0))
        .when(column.isin("Not Applicable", "not applicable"), F.lit(None))
        .when(column.contains("%"), cleaned_percent.cast("double") / 100.0)
        .otherwise(cleaned_numeric)
        .cast("double")
    )

def parse_string_to_boolean(col_name: str):
    """
    Converte coluna de string para booleano.
    Trata valores típicos: "Y"/"Yes"/"1"/"True" → true, "N"/"No"/"0"/"False" → false.
    """
    col_upper = F.upper(F.col(col_name))
    return F.when(
        col_upper.isin("Y", "YES", "1", "TRUE", "COVERED", "NEW"),
        True
    ).when(
        col_upper.isin("N", "NO", "0", "FALSE", "NOT COVERED", "EXISTING"),
        False
    ).otherwise(
        None
    ).cast("boolean")

def parse_string_to_timestamp(col_name: str):
    """
    Converte coluna de string para timestamp.
    Tenta múltiplos formatos comuns.
    """
    return F.to_timestamp(F.col(col_name), "MM/dd/yyyy").cast("timestamp")

def parse_string_to_integer(col_name: str):
    """
    Converte coluna de string para inteiro.
    Trata "Not Applicable" → null.
    """
    return F.when(
        F.col(col_name).isin("Not Applicable", "not applicable"),
        F.lit(None).cast("integer")
    ).otherwise(
        F.col(col_name).cast("integer")
    ).cast("integer")

def deduplicate_by_version(df, partition_cols):
    """
    Deduplica DataFrame por partition_cols + PlanId, mantendo maior VersionNum.
    """
    window = Window.partitionBy(*partition_cols,"PlanId").orderBy(F.col("VersionNum").desc())
    return df.withColumn("rn", F.row_number().over(window)).filter("rn = 1").drop("rn")

def apply_transformations(table_name: str, df):
    """
    Aplica transformações específicas por tabela baseadas nas recomendações.
    """

    # Converter colunas de string para double
    double_cols = [
        "CopayInnTier1", "CopayInnTier2", "CopayOutofNet",
        "CoinsInnTier1", "CoinsInnTier2", "CoinsOutofNet",
        "IndividualRate", "IndividualTobaccoRate", "Couple",
        "PrimarySubscriberAndOneDependent", "PrimarySubscriberAndTwoDependents",
        "PrimarySubscriberAndThreeOrMoreDependents", "CoupleAndOneDependent",
        "CoupleAndTwoDependents", "CoupleAndThreeOrMoreDependents",
        "AvCalculatorOutputNumber", "DEHBCombInnOonFamilyMOOP", "DEHBCombInnOonIndividualMOOP",
        "DEHBDedCombInnOonFamily", "DEHBDedCombInnOonIndividual", "DEHBDedInnTier1Coinsurance",
        "DEHBDedInnTier1Family", "DEHBDedInnTier1Individual", "DEHBDedInnTier2Coinsurance",
        "DEHBDedInnTier2Family", "DEHBDedInnTier2Individual", "DEHBDedOutOfNetFamily",
        "DEHBDedOutOfNetIndividual", "DEHBInnTier1FamilyMOOP", "DEHBInnTier1IndividualMOOP",
        "DEHBOutOfNetFamilyMOOP", "DEHBOutOfNetIndividualMOOP", "EHBPediatricDentalApportionmentQuantity",
        "EHBPercentPremiumS4", "EHBPercentTotalPremium", "FirstTierUtilization", "HPID",
        "IndianPlanVariationEstimatedAdvancedPaymentAmountPerEnrollee", "IssuerActuarialValue",
        "MEHBCombInnOonFamilyMOOP", "MEHBCombInnOonIndividualMOOP", "MEHBDedCombInnOonFamily",
        "MEHBDedCombInnOonIndividual", "MEHBDedInnTier1Coinsurance", "MEHBDedInnTier1Family",
        "MEHBDedInnTier1Individual", "MEHBDedInnTier2Coinsurance", "MEHBDedInnTier2Family",
        "MEHBDedInnTier2Individual","MEHBDedOutOfNetFamily", "MEHBDedOutOfNetIndividual",
        "MEHBInnTier1FamilyMOOP", "MEHBInnTier1IndividualMOOP", "MEHBInnTier2FamilyMOOP",
        "MEHBInnTier2IndividualMOOP", "MEHBOutOfNetFamilyMOOP", "MEHBOutOfNetIndividualMOOP",
        "SBCHavingDiabetesCoinsurance", "SBCHavingDiabetesCopayment", "SBCHavingDiabetesDeductible",
        "SBCHavingDiabetesLimit", "SBCHavingaBabyCoinsurance", "SBCHavingaBabyCopayment", "SBCHavingaBabyDeductible",
        "SBCHavingaBabyLimit", "SecondTierUtilization", "SpecialtyDrugMaximumCoinsurance", "TEHBCombInnOonFamilyMOOP",
        "TEHBCombInnOonIndividualMOOP", "TEHBDedCombInnOonFamily", "TEHBDedCombInnOonIndividual", "TEHBDedInnTier1Coinsurance",
        "TEHBDedInnTier1Family", "TEHBDedInnTier1Individual", "TEHBDedInnTier2Coinsurance", "TEHBDedInnTier2Family",
        "TEHBDedInnTier2Individual", "TEHBDedOutOfNetFamily", "TEHBDedOutOfNetIndividual", "TEHBInnTier1FamilyMOOP",
        "TEHBInnTier1IndividualMOOP", "TEHBInnTier2FamilyMOOP", "TEHBInnTier2IndividualMOOP", "TEHBOutOfNetFamilyMOOP",
        "TEHBOutOfNetIndividualMOOP"
    ]
    for col in double_cols:
        if col in df.columns:
            df = df.withColumn(col, parse_string_to_double(col))

    # Converter ImportDate para timestamp
    timestamp_cols = [
        "ImportDate", "RateEffectiveDate", "RateExpirationDate",
        "PlanEffictiveDate", "PlanExpirationDate"]
    for col in timestamp_cols:
        if col in df.columns:
            df = df.withColumn(col, parse_string_to_timestamp(col))

    # Converter colunas booleanas (IsXXX)
    bool_cols = [
        "IsCovered", "IsEHB", "IsExclFromInnMOOP",
        "IsExclFromOonMOOP", "IsStateMandate",
        "IsSubjToDedTier1", "IsSubjToDedTier2", "QuantLimitOnSvc",
        "DomesticPartnerAsSpouseIndicator", "SameSexPartnerAsSpouseIndicator",
        "DentalOnlyPlan", "DentalPlan", "MultistatePlan_2014", "MultistatePlan_2015",
        "MultistatePlan_2016", "CoverEntireState", "PartialCounty", "IsHSAEligible", 
        "IsNewPlan", "IsNoticeRequiredForPregnancy", "IsReferralRequiredForSpecialist",
        "UniquePlanDesign", "WellnessProgramOffered"
    ]
    for col in bool_cols:
        if col in df.columns:
            df = df.withColumn(col, parse_string_to_boolean(col))

    # Converter colunas para integer (com tratamento de "Not Applicable")
    int_cols = [
        "LimitQty", "BusinessYear", "MinimumStay",
        "MinimumTobaccoFreeMonthsRule", "DependentMaximumAgRule", "ChildAdultOnly_2014",
        "CrosswalkLevel", "ReasonForCrosswalk", "ChildAdultOnly_2015", "ChildAdultOnly_2016",
        "Age", "BeginPrimaryCareCostSharingAfterNumberOfVisits", "BeginPrimaryCareDeductibleCoinsuranceAfterNumberOfCopays",
        "BenefitPackageId", "InpatientCopaymentMaximumDays", "MedicalDrugDeductiblesIntegrated",
        "MedicalDrugMaximumOutofPocketIntegrated", "MultipleInNetworkTiers", "NationalNetwork",
        "OutOfCountryCoverage","OutOfServiceAreaCoverage"
    ]
    for col in int_cols:
        if col in df.columns:
            df = df.withColumn(col, parse_string_to_integer(col))

    # Deduplicação geral se aplicável
    if all(col in df.columns for col in ["BenefitName", "BusinessYear", "VersionNum"]):
        df = deduplicate_by_version(df, ["BenefitName", "BusinessYear", "VersionNum"])

    return df

In [6]:
# ##################################################################################################
# Descoberta das tabelas na Bronze

tables_df = wr.catalog.tables(database=BRONZE_DATABASE)
all_table_names = sorted(
    [t for t in tables_df["Table"].tolist() if not t.lower().startswith("raw_")]
)

logger.info("Total de tabelas na Bronze (excluindo raw_): %d", len(all_table_names))
print(all_table_names)


[2026-04-12T21:44:42] INFO — Total de tabelas na Bronze (excluindo raw_): 8


['benefits_cost_sharing', 'business_rules', 'crosswalk2015', 'crosswalk2016', 'network', 'plan_attributes', 'rate', 'service_area']


In [7]:
# ##################################################################################################
# Criar database Silver se não existir

try:
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {SILVER_DATABASE}")
    logger.info("Database '%s' criado/verificado.", SILVER_DATABASE)
except Exception as exc:
    logger.error("Erro ao criar database '%s': %s", SILVER_DATABASE, exc)
    raise

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/share/aws/aws-java-sdk-v2/aws-sdk-java-bundle-2.29.52.jar!/software/amazon/awssdk/thirdparty/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/share/aws/glue-pds/jars/bundle-2.24.6.jar!/software/amazon/awssdk/thirdparty/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
[2026-04-12T21:44:47] INFO — Database 'eedb015_silver' criado/verificado.


In [9]:
# ##################################################################################################
# Processamento: ler Bronze, transformar, escrever Silver

tables_to_process = all_table_names

if LOCAL_TABLE_LIMIT:
    logger.info("Ambiente local: limitando a %d tabela(s).", LOCAL_TABLE_LIMIT)
    tables_to_process = tables_to_process[:LOCAL_TABLE_LIMIT]

tables_ok = []
tables_failed = []

for table_name in tables_to_process:
    bronze_table = f"{BRONZE_DATABASE}.{table_name}"
    silver_table = f"{SILVER_DATABASE}.{table_name}"
    logger.info("Processando tabela '%s' → '%s'…", bronze_table, silver_table)

    try:
        # Ler da Bronze
        df = spark.table(bronze_table)
        logger.info("  Lida: %d coluna(s), %d linha(s) estimadas.", len(df.columns), df.count())

        # Aplicar transformações
        df_transformed = apply_transformations(table_name, df)
        logger.info("  Transformada: %d coluna(s).", len(df_transformed.columns))

        # Deletar tabela anterior para evitar conflito de schema (Iceberg força tipo original)
        try:
            spark.sql(f"DROP TABLE IF EXISTS {silver_table}")
            logger.info("  Tabela anterior removida.")
        except Exception as e:
            logger.warning("  Aviso ao remover tabela anterior: %s", e)

        # Escrever na Silver
        df_transformed.writeTo(silver_table).createOrReplace()
        logger.info("  Tabela '%s' criada/atualizada com sucesso.", silver_table)
        
        # Verificar se a tabela foi criada corretamente
        try:
            df_final = spark.table(silver_table)
            logger.info("  Verificação: Tabela criada com %d coluna(s).", len(df_final.columns))

        except Exception as e:
            logger.warning("  Aviso na verificação da tabela: %s", e)
        
        tables_ok.append(silver_table)

    except Exception as exc:
        logger.error("  ERRO ao processar tabela '%s': %s", silver_table, exc, exc_info=True)
        tables_failed.append(silver_table)

# Resumo
logger.info("=" * 60)
logger.info("Processamento concluído.")
logger.info("  Tabelas OK    : %d — %s", len(tables_ok), tables_ok)
if tables_failed:
    logger.error("  Tabelas FALHA : %d — %s", len(tables_failed), tables_failed)

[2026-04-12T21:46:27] INFO — Processando tabela 'eedb015_bronze.benefits_cost_sharing' → 'eedb015_silver.benefits_cost_sharing'…


26/04/12 21:46:32 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[2026-04-12T21:46:34] INFO —   Lida: 34 coluna(s), 5048468 linha(s) estimadas.
[2026-04-12T21:46:36] INFO —   Transformada: 34 coluna(s).
[2026-04-12T21:46:38] INFO —   Tabela anterior removida.
[2026-04-12T21:49:21] INFO —   Tabela 'eedb015_silver.benefits_cost_sharing' criada/atualizada com sucesso.
[2026-04-12T21:49:22] INFO —   Verificação: Tabela criada com 34 coluna(s).
[2026-04-12T21:49:22] INFO — Processando tabela 'eedb015_bronze.business_rules' → 'eedb015_silver.business_rules'…
[2026-04-12T21:49:24] INFO —   Lida: 25 coluna(s), 21085 linha(s) estimadas.
[2026-04-12T21:49:24] INFO —   Transformada: 25 coluna(s).
[2026-04-12T21:49:26] INFO —   Tabela anterior removida.
[2026-04-12T21:49:33] INFO —   Tabela 'eedb015_silver.business_rules' criada/atualizada com sucesso.
[2026-04-12T21:49:34] I